# C1 — BVP + EDA Exploratory Data Analysis

This notebook contains the exploratory data analysis extracted from `c1_bvp_eda_full_pipeline.ipynb`. It analyses the cached, preprocessed WESAD wrist BVP + EDA windows without running model training.

The source full-pipeline notebook remains unchanged.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
print('EDA dependencies ready.')

## 2. Load the preprocessed BVP + EDA dataset

The preprocessing pipeline stores engineered features (`X_feat`), aligned raw `[EDA, BVP]` windows (`X_raw`), binary labels, subject IDs, and feature names in an NPZ file.

In [ ]:
# Find the repository root whether Jupyter starts at the project root or this folder.
search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (root for root in search_roots if (root / 'data/c1_stress_detection_bvp').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find the project data directory. Start Jupyter inside the repository.'
    )

DATA_ROOT = PROJECT_ROOT / 'data/c1_stress_detection_bvp/BVP_EDA/processed'
dataset_candidates = sorted(DATA_ROOT.rglob('dataset_bvp_eda.npz'))
if not dataset_candidates:
    raise FileNotFoundError(
        f'No dataset_bvp_eda.npz found below {DATA_ROOT}. Run preprocessing first.'
    )

# Both exported runs contain the same dataset schema; use the first available copy.
DATASET_PATH = dataset_candidates[0]
REPORT_DIR = PROJECT_ROOT / 'reports/c1_stress_detection_bvp/BVP_EDA/eda_outputs'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(DATASET_PATH, allow_pickle=True) as data:
    X_feat = data['X_feat']
    X_raw = data['X_raw']
    y_all = data['y'].astype(np.int64)
    groups = data['groups'].astype(np.int64)
    FEATURE_COLS = [str(name) for name in data['feature_cols']]

print(f'Loaded: {DATASET_PATH}')
print(f'X_feat : {X_feat.shape}')
print(f'X_raw  : {X_raw.shape} (channels: [EDA, BVP])')
print(
    f'y      : {y_all.shape}  baseline={int((y_all == 0).sum())}, '
    f'stress={int((y_all == 1).sum())} ({100 * (y_all == 1).mean():.1f}% stress)'
)
print(f'groups : {len(np.unique(groups))} subjects')

## 3. Dataset overview and sanity checks

In [ ]:
df_feat = pd.DataFrame(X_feat, columns=FEATURE_COLS)
df_feat['label'] = y_all
df_feat['subject'] = groups

display(df_feat.head())
display(df_feat[FEATURE_COLS].describe().T)

print('Missing values per feature (NaN = too few beats in that window)')
missing = df_feat[FEATURE_COLS].isna().mean().sort_values(ascending=False)
print((missing[missing > 0] * 100).round(2).to_string() if (missing > 0).any() else '  none')

print('\nDegenerate features (zero variance)')
zero_variance = [
    column for column in FEATURE_COLS
    if df_feat[column].std(skipna=True) < 1e-10
]
print(f'  {zero_variance}' if zero_variance else '  none')

## 4. Per-subject class balance

In [ ]:
class_balance = (
    df_feat.groupby('subject')['label']
    .agg(n='size', baseline=lambda values: int((values == 0).sum()),
         stress=lambda values: int((values == 1).sum()), stress_rate='mean')
)
display(class_balance.round(3))

ax = class_balance[['baseline', 'stress']].plot(
    kind='bar', stacked=True, figsize=(12, 5), color=['#4C72B0', '#C44E52']
)
ax.set(title='BVP + EDA windows per subject', xlabel='Subject', ylabel='Windows')
ax.legend(title='Class', labels=['Baseline', 'Stress'])
plt.tight_layout()
plt.savefig(REPORT_DIR / 'class_balance_by_subject.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Feature correlation and collinearity

In [ ]:
correlation = df_feat[FEATURE_COLS].corr()
absolute_correlation = correlation.abs()

print('Highly collinear feature pairs (|r| > 0.98)')
high_pairs = []
for index, first in enumerate(FEATURE_COLS):
    for second in FEATURE_COLS[index + 1:]:
        value = absolute_correlation.loc[first, second]
        if value > 0.98:
            high_pairs.append((first, second, value))
            print(f'  {first:22s} <-> {second:22s}  r = {value:.4f}')
if not high_pairs:
    print('  none')

plt.figure(figsize=(14, 11))
sns.heatmap(correlation, cmap='vlag', center=0, vmin=-1, vmax=1, square=True)
plt.title('BVP + EDA feature correlation matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Feature distributions by class

Values are clipped to each feature's 1st–99th percentile for readable plots; the underlying dataset is not modified.

In [ ]:
n_columns = 5
n_rows = int(np.ceil(len(FEATURE_COLS) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(20, 3 * n_rows))
axes = np.asarray(axes).ravel()

for axis, feature in zip(axes, FEATURE_COLS):
    for label, name, color in [
        (0, 'Baseline', '#4C72B0'),
        (1, 'Stress', '#C44E52'),
    ]:
        values = df_feat.loc[df_feat.label == label, feature].dropna()
        if len(values):
            clipped = np.clip(values, values.quantile(0.01), values.quantile(0.99))
            axis.hist(clipped, bins=40, alpha=0.55, label=name, color=color, density=True)
    axis.set_title(feature, fontsize=9)
    axis.tick_params(labelsize=7)
    axis.set_yticks([])

for axis in axes[len(FEATURE_COLS):]:
    axis.axis('off')
axes[0].legend(fontsize=8)
plt.suptitle('Feature distributions by class (1st–99th percentile)', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. EDA-specific feature summary

In [ ]:
EDA_FEATURES = [feature for feature in FEATURE_COLS if feature.startswith('eda_')]
eda_summary = (
    df_feat.groupby('label')[EDA_FEATURES]
    .mean()
    .rename(index={0: 'Baseline', 1: 'Stress'})
    .T
)
eda_summary['stress_minus_baseline'] = eda_summary['Stress'] - eda_summary['Baseline']
display(eda_summary.sort_values('stress_minus_baseline', key=np.abs, ascending=False))

print(f'Plots saved to: {REPORT_DIR}')